In [0]:
%sql
-- Estes pedidos vem com link para filtro de categoria / produto
-- Agregados por ano/mes e status (pq tem muitos "cancelados")
CREATE OR REPLACE TABLE workspace.gold.gold_comercial_produto AS
SELECT
  DATE_FORMAT(p.order_date, 'yyyy-MM')              AS order_date,
  p.status_order,
  pr.category,
  pr.name,
  SUM(p.gross_amount_num)                           AS receita_bruta,
  SUM(p.net_amount)                                 AS receita_liquida,
  SUM(p.discount_amount)                            AS desconto_total,
  ROUND(
    SUM(p.discount_amount) * 100.0
    / NULLIF(SUM(p.gross_amount_num), 0), 2
  )                                                 AS perc_desconto,
  AVG(p.gross_amount_num)                           AS ticket_medio,
  SUM(i.quantity)                                   AS total_itens
FROM workspace.silver.tb_pedidos_cabecalho p
LEFT JOIN workspace.silver.tb_pedidos_itens i  ON p.order_id     = i.order_id
LEFT JOIN workspace.silver.dim_produto      pr ON i.product_code = pr.product_id
GROUP BY
  DATE_FORMAT(p.order_date, 'yyyy-MM'),
  p.status_order,
  pr.category,
  pr.name

In [0]:
%sql
-- Estes pedidos vem com link para filtro de cliente, canal e/ou regiao
-- Agregados por ano/mes e status (pq tem muitos "cancelados")
CREATE OR REPLACE TABLE workspace.gold.gold_comercial_clientes AS
SELECT
  DATE_FORMAT(p.order_date, 'yyyy-MM')                               AS ano_mes,
  p.status_order,
  v.canal_id,
  v.regional_code,
  c.customer_id,
  c.nome_cliente,
  c.segmento,
  c.porte,
  c.cidade,
  c.estado,
  c.status_cliente,
  COUNT(DISTINCT p.order_id)                                         AS total_pedidos,
  SUM(p.gross_amount_num)                                            AS receita_bruta,
  SUM(p.net_amount)                                                  AS receita_liquida,
  SUM(p.discount_amount)                                             AS desconto_total,
  ROUND(
    SUM(p.discount_amount) * 100.0
    / NULLIF(SUM(p.gross_amount_num), 0), 2
  )                                                                  AS perc_desconto,
  AVG(p.gross_amount_num)                                            AS ticket_medio
FROM workspace.silver.tb_pedidos_cabecalho p 
LEFT JOIN workspace.silver.dim_clientes    c  ON c.customer_id = p.customer_code
LEFT JOIN workspace.silver.dim_vendedores  v  ON p.seller_id    = v.seller_id
GROUP BY
  DATE_FORMAT(p.order_date, 'yyyy-MM'),
  v.canal_id,
  v.regional_code,
  c.customer_id,
  c.nome_cliente,
  c.segmento,
  c.porte,
  c.cidade,
  c.estado,
  c.status_cliente,
  c.data_cadastro,
  p.status_order

In [0]:
%sql
-- Estas entregas de pedidos vem com link para filtro de carrier_name, carrier_mode, delivery_status, state, city.
-- Agregados por ano/mes e pelos filtros + status (pq tem muitos "cancelados").
CREATE OR REPLACE TABLE workspace.gold.gold_entrega_pedido AS
SELECT
  DATE_FORMAT(p.order_date, 'yyyy-MM')                    AS ano_mes,
  p.status_order,
  e.carrier_name,
  e.carrier_mode,
  CASE
    WHEN LOWER(e.delivery_status) = 'delivered'   THEN 'Entregue'
    WHEN LOWER(e.delivery_status) = 'in_transit'  THEN 'Em Trânsito'
    WHEN LOWER(e.delivery_status) = 'atrasado'    THEN 'Atrasado'
    WHEN LOWER(e.delivery_status) = 'cancelled'   THEN 'Cancelado'
    ELSE NULL
  END                                                               AS delivery_status,
  e.state,
  e.city,
  COUNT(DISTINCT e.order_ref)                                       AS total_entregas,
  SUM(e.cost)                                                       AS custo_total_frete,
  AVG(e.cost)                                                       AS custo_medio_frete,
  AVG(DATEDIFF(e.shipped_at, p.order_date))                         AS media_dias_ate_expedicao,
  AVG(DATEDIFF(e.delivered_at, e.shipped_at))                       AS media_dias_transito,
  AVG(DATEDIFF(e.delivered_at, p.order_date))                       AS media_dias_ciclo_total,
  AVG(DATEDIFF(e.delivered_at, p.promised_date))                    AS media_dias_atraso,
  SUM(CASE WHEN e.delivered_at > p.promised_date THEN 1 ELSE 0 END) AS total_atrasados,
  ROUND(
    SUM(CASE WHEN e.delivered_at > p.promised_date THEN 1 ELSE 0 END) * 100.0
    / NULLIF(COUNT(DISTINCT e.order_ref), 0), 2
  )                                                                  AS perc_atrasados,
  SUM(p.gross_amount_num)                                            AS receita_bruta,
  SUM(p.net_amount)                                                  AS receita_liquida
FROM workspace.silver.tb_entrega               e
LEFT JOIN workspace.silver.tb_pedidos_cabecalho p ON e.order_ref = p.order_id
GROUP BY
  DATE_FORMAT(p.order_date, 'yyyy-MM'),
  p.status_order,
  e.carrier_name,
  e.carrier_mode,
  LOWER(e.delivery_status),
  e.state,
  e.city

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_atendimentos_pedido AS
SELECT
  DATE_FORMAT(a.created_at, 'yyyy-MM')                              AS ano_mes,
  CASE
    WHEN a.event_type = 'Delay'          THEN 'Atraso'
    WHEN a.event_type = 'Refund'         THEN 'Reembolso'
    WHEN a.event_type = 'Troca'          THEN 'Troca'
    WHEN a.event_type = 'Complaint'      THEN 'Reclamação'
    WHEN a.event_type = 'Cancel_request' THEN 'Cancelamento'
    ELSE NULL
  END                                                               AS event_type,
  a.severity,
  CASE
    WHEN a.status = 'Open'   THEN 'Aberto'
    WHEN a.status = 'Closed' THEN 'Encerrado'
    ELSE NULL
  END                                                               AS status_atendimento,
  p.seller_id,
  p.status_order,
  COUNT(DISTINCT a.ticket_id)                                       AS total_tickets,
  COUNT(CASE WHEN a.severity = 'High'   THEN 1 END)                 AS tickets_high,
  COUNT(CASE WHEN a.severity = 'Medium' THEN 1 END)                 AS tickets_medium,
  COUNT(CASE WHEN a.severity = 'Low'    THEN 1 END)                 AS tickets_low,
  COUNT(CASE WHEN a.status   = 'Open'   THEN 1 END)                 AS tickets_abertos,
  COUNT(CASE WHEN a.status   = 'Closed' THEN 1 END)                 AS tickets_encerrados,

  ROUND(
    COUNT(CASE WHEN a.status = 'Open' THEN 1 END) * 100.0
    / NULLIF(COUNT(DISTINCT a.ticket_id), 0), 2
  )                                                                  AS perc_abertos,
  COUNT(
    CASE WHEN a.status = 'Open'
          AND DATEDIFF(CURRENT_DATE(), a.created_at) > 7 THEN 1 END
  )                                                                  AS tickets_criticos,
  AVG(DATEDIFF(a.created_at, p.order_date))                          AS media_dias_pedido_ate_ticket,
  SUM(p.gross_amount_num)                                            AS receita_bruta,
  SUM(p.net_amount)                                                  AS receita_liquida,
  SUM(p.discount_amount)                                             AS desconto_total,
  COUNT(DISTINCT p.order_id)                                         AS total_pedidos_afetados
FROM workspace.silver.tb_atendimentos           a
LEFT JOIN workspace.silver.tb_pedidos_cabecalho p ON a.order_id = p.order_id
GROUP BY
  DATE_FORMAT(a.created_at, 'yyyy-MM'),
  CASE
    WHEN a.event_type = 'Delay'          THEN 'Atraso'
    WHEN a.event_type = 'Refund'         THEN 'Reembolso'
    WHEN a.event_type = 'Troca'          THEN 'Troca'
    WHEN a.event_type = 'Complaint'      THEN 'Reclamação'
    WHEN a.event_type = 'Cancel_request' THEN 'Cancelamento'
    ELSE NULL
  END,
  a.severity,
  CASE
    WHEN a.status = 'Open'   THEN 'Aberto'
    WHEN a.status = 'Closed' THEN 'Encerrado'
    ELSE NULL
  END,
  p.seller_id,
  p.status_order